In [1]:
%reset -f

In [2]:
%reset -f

In [3]:
%reset -f

In [4]:
# stop 

In [5]:
!nvidia-smi

Sun Oct 18 09:17:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   49C    P8              3W /   95W |     952MiB /   8188MiB |      6%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Setup

In [6]:
import os 
os.chdir('D:/College/AI/LLM Learning/Module 0/Module-0-Transformers-from-Scratch')

os.getcwd()

'D:\\College\\AI\\LLM Learning\\Module 0\\Module-0-Transformers-from-Scratch'

In [7]:
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)

# Packages

In [8]:
# =====================================================================
# 1. Standard Library & Utilities
# =====================================================================
import os
import sys
import math
import time
import json
import random
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from collections import Counter
from datasets import load_dataset, Dataset

In [9]:
# =====================================================================
# 2. General Data Manipulation, Science & Visualization
# =====================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [10]:
# =====================================================================
# 3. Deep Learning & PyTorch Core
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR

In [11]:
# =====================================================================
# 4. Tokenization & NLP Metrics
# =====================================================================
# Custom subword tokenization from scratch (no pretrained weights)
# If using Hugging Face's tokenizers library:
try:
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
except ImportError:
    pass

# Metric computation (ROUGE scores for abstractive summarization)
try:
    from rouge_score import rouge_scorer
except ImportError:
    pass

In [12]:
# =====================================================================
# 5. Device Verification & Seeding Utility
# =====================================================================
def set_seed(seed) -> None:
    # random.seed(seed)
    # np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(seed=seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

# Hardware / Device Status
print("-" * 50)
try:
    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU only"
    print(f"CUDA Available: {cuda_available}")
    print(f"Active Device:  {device_name}")
except ImportError:
    print("PyTorch not installed.")
print("-" * 50)

Using device: cuda
Device Name: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.0 GB
--------------------------------------------------
CUDA Available: True
Active Device:  NVIDIA GeForce RTX 4060 Laptop GPU
--------------------------------------------------


# Load Dataset

In [13]:
# Define dataset paths relative to root directory
RAW_DATA_DIR = os.path.join(os.getcwd(), "data", "raw")
DATASET_PATH = os.path.join(RAW_DATA_DIR, "summarization_dataset.json")
HF_DATASET_NAME = "abhiprd2000/nlp-benchmark-suite"

os.makedirs(RAW_DATA_DIR, exist_ok=True)

try:
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f"Local file does not exist at {DATASET_PATH}")
    
    # 1. Read existing local dataset
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        records = json.load(f)
    dataset = Dataset.from_list(records)
    print(f"Successfully loaded from local file ({len(dataset):,} examples)!")

except Exception as e:
    print(f"Downloading from Hugging Face: {HF_DATASET_NAME} ...")
    
    # 2. Download text_summarization split from Hugging Face
    hf_data = load_dataset(HF_DATASET_NAME, data_files={"train": "text_summarization.json"})
    dataset = hf_data['train']
    
    # 3. Save to data/raw/
    records = [dict(item) for item in dataset]
    with open(DATASET_PATH, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
        
    print(f"File size: {round(os.path.getsize(DATASET_PATH) / 1024, 2)} KB")

# Display schema & sample
print("\n" + "=" * 50)
print(f"Total Examples: {len(dataset):,}")
print(f"Columns: {dataset.column_names}")
print("=" * 50)
sample = dataset[0]
print(f"[article]:\n{sample['article'][:200]}...\n")
print(f"[summary]:\n{sample['summary']}")

File size: 1386.38 KB

Total Examples: 1,500
Columns: ['id', 'article', 'summary', 'article_length', 'summary_length', 'compression_ratio']
[article]:
Scientists at MIT have developed a new type of neural network architecture that processes text 40% faster than existing transformer models while maintaining the same accuracy. The breakthrough, publis...

[summary]:
MIT researchers developed a neural network 40% faster than transformers, using sparse attention to cut computational costs while maintaining accuracy.


In [14]:
# try:
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Local file does not exist at {DATASET_PATH}")

# 1. Read existing local dataset
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)
dataset = Dataset.from_list(records)
print(f"Successfully loaded from local file ({len(dataset):,} examples)!")

# # except Exception as e:
# print(f"Downloading from Hugging Face: {HF_DATASET_NAME} ...")

# # 2. Download text_summarization split from Hugging Face
# hf_data = load_dataset(HF_DATASET_NAME, data_files={"train": "text_summarization.json"})
# dataset = hf_data['train']

# # 3. Save to data/raw/
# records = [dict(item) for item in dataset]
# with open(DATASET_PATH, "w", encoding="utf-8") as f:
#     json.dump(records, f, indent=2, ensure_ascii=False)
    
print(f"File size: {round(os.path.getsize(DATASET_PATH) / 1024, 2)} KB")

AttributeError: type object 'Dataset' has no attribute 'from_list'